# Analisis de conjunto de datos de viajes

Análisis de conjunto de datos de viajes en omnibus en Montevideo que provee la Intendencia de Montevideo de manera pública. En particular, se usan los datos de enero de 2022:
[https://ckan.montevideo.gub.uy/dataset/viajes-realizados-en-los-omnibus-del-sistema-de-transporte-metropolitano-stm]

In [2]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 13.4 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from unidecode import unidecode
import os

## Descarga y carga de los datos

In [ ]:
descargar_datasets(datasets="viajes", carpeta="../datasets")

Descargando: u0u-R-HUTN2ws0st23-2ow...
✅ u0u-R-HUTN2ws0st23-2ow guardado.

Descargando: descripcion_de_los_atributos-1.ods...
✅ descripcion_de_los_atributos-1.ods guardado.


Los datasets se encuentran disponibles en la ruta:  /home/carmen/fing/2025/aagrafos-labs/proyecto/analisis_datasets/../datasets


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = "/content/drive/MyDrive/AAGrafos/Proyecto/horario_teorico/viajes_stm_032022.csv"
df = pd.read_csv(path, encoding="utf-8")

display(df.head())

Para evitar problemas de memoria al cargar archivos CSV grandes, podemos determinar el número total de filas y luego usar el parámetro `nrows` de `pd.read_csv` para cargar solo una porción del archivo. Primero, contemos las líneas:

In [3]:
import pandas as pd
import os

# La variable 'path' debe estar definida para su uso.
# Si no se ha ejecutado previamente, la definimos aquí.
path = "/content/drive/MyDrive/AAGrafos/Proyecto/horario_teorico/viajes_stm_032022.csv"

# Contar el número total de líneas en el archivo CSV (excluyendo el encabezado)
# Esto se hace de forma eficiente sin cargar todo el archivo en memoria

# Obtener el nombre del archivo de la ruta completa
filename = os.path.basename(path)
print(f"Contando líneas en el archivo: {filename}")

# Usar un enfoque más robusto para contar líneas, útil para archivos muy grandes
num_lines = 0
with open(path, 'r', encoding='utf-8', errors='ignore') as f:
    # Saltar el encabezado si existe
    try:
        next(f)
    except StopIteration:
        pass # El archivo está vacío o solo tiene un encabezado

    for line in f:
        num_lines += 1

print(f"El archivo CSV tiene un total de {num_lines} filas de datos (sin contar el encabezado).")

Contando líneas en el archivo: viajes_stm_032022.csv
El archivo CSV tiene un total de 25610312 filas de datos (sin contar el encabezado).


Ahora que conocemos el número total de filas, podemos cargar aproximadamente el 50% de ellas usando `nrows`.

In [4]:
# Calcular el 50% de las filas
half_num_lines = num_lines // 2

print(f"Cargando las primeras {half_num_lines} filas (aproximadamente el 50%)...")

# Cargar el CSV con el número limitado de filas
df_half = pd.read_csv(path, encoding="utf-8", nrows=half_num_lines)

print(f"Se cargaron {len(df_half)} filas.")
display(df_half.head())
display(df_half.info())

Cargando las primeras 12805156 filas (aproximadamente el 50%)...
Se cargaron 12805156 filas.


,id_viaje,con_tarjeta,fecha_evento,tipo_viaje,descripcion_tipo_viaje,grupo_usuario,descripcion_grupo_usuario,grupo_usuario_especifico,descripcion_grupo_usuario_espe,ordinal_de_tramo,cantidad_pasajeros,codigo_parada_origen,cod_empresa,descrip_empresa,linea_codigo,dsc_linea,sevar_codigo
0,1,1,2022-03-01T00:00:02.000-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,1642,50,C.U.T.C.S.A.,446,151,3363
1,2,1,2022-03-01T00:00:06.000-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,1642,50,C.U.T.C.S.A.,446,151,3363
2,3,1,2022-03-01T00:02:01.000-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,3713,50,C.U.T.C.S.A.,73,158,8546
3,4,1,2022-03-01T00:09:40.000-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,3217,50,C.U.T.C.S.A.,26,102,269
4,5,0,2022-03-01T00:07:12.000-03:00,3.0,CENTRICO,0,NO CORRESPONDE,0,NO CORRESPONDE,0,1,4014,50,C.U.T.C.S.A.,98,60,1483


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12805156 entries, 0 to 12805155
Data columns (total 17 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   id_viaje                        int64  
 1   con_tarjeta                     int64  
 2   fecha_evento                    object 
 3   tipo_viaje                      float64
 4   descripcion_tipo_viaje          object 
 5   grupo_usuario                   int64  
 6   descripcion_grupo_usuario       object 
 7   grupo_usuario_especifico        int64  
 8   descripcion_grupo_usuario_espe  object 
 9   ordinal_de_tramo                int64  
 10  cantidad_pasajeros              int64  
 11  codigo_parada_origen            int64  
 12  cod_empresa                     int64  
 13  descrip_empresa                 object 
 14  linea_codigo                    int64  
 15  dsc_linea                       object 
 16  sevar_codigo                    int64  
dtypes: float64(1), int64(10),

None

In [5]:
df=df_half

In [6]:
df["fecha_evento"] = pd.to_datetime(df["fecha_evento"])

df.head()

,id_viaje,con_tarjeta,fecha_evento,tipo_viaje,descripcion_tipo_viaje,grupo_usuario,descripcion_grupo_usuario,grupo_usuario_especifico,descripcion_grupo_usuario_espe,ordinal_de_tramo,cantidad_pasajeros,codigo_parada_origen,cod_empresa,descrip_empresa,linea_codigo,dsc_linea,sevar_codigo
0,1,1,2022-03-01 00:00:02-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,1642,50,C.U.T.C.S.A.,446,151,3363
1,2,1,2022-03-01 00:00:06-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,1642,50,C.U.T.C.S.A.,446,151,3363
2,3,1,2022-03-01 00:02:01-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,3713,50,C.U.T.C.S.A.,73,158,8546
3,4,1,2022-03-01 00:09:40-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,3217,50,C.U.T.C.S.A.,26,102,269
4,5,0,2022-03-01 00:07:12-03:00,3.0,CENTRICO,0,NO CORRESPONDE,0,NO CORRESPONDE,0,1,4014,50,C.U.T.C.S.A.,98,60,1483


In [7]:
print("Cantidad de datos:",len(df))
display(df.head())
display(df.info())

Cantidad de datos: 12805156


,id_viaje,con_tarjeta,fecha_evento,tipo_viaje,descripcion_tipo_viaje,grupo_usuario,descripcion_grupo_usuario,grupo_usuario_especifico,descripcion_grupo_usuario_espe,ordinal_de_tramo,cantidad_pasajeros,codigo_parada_origen,cod_empresa,descrip_empresa,linea_codigo,dsc_linea,sevar_codigo
0,1,1,2022-03-01 00:00:02-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,1642,50,C.U.T.C.S.A.,446,151,3363
1,2,1,2022-03-01 00:00:06-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,1642,50,C.U.T.C.S.A.,446,151,3363
2,3,1,2022-03-01 00:02:01-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,3713,50,C.U.T.C.S.A.,73,158,8546
3,4,1,2022-03-01 00:09:40-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,3217,50,C.U.T.C.S.A.,26,102,269
4,5,0,2022-03-01 00:07:12-03:00,3.0,CENTRICO,0,NO CORRESPONDE,0,NO CORRESPONDE,0,1,4014,50,C.U.T.C.S.A.,98,60,1483


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12805156 entries, 0 to 12805155
Data columns (total 17 columns):
 #   Column                          Dtype                    
---  ------                          -----                    
 0   id_viaje                        int64                    
 1   con_tarjeta                     int64                    
 2   fecha_evento                    datetime64[ns, UTC-03:00]
 3   tipo_viaje                      float64                  
 4   descripcion_tipo_viaje          object                   
 5   grupo_usuario                   int64                    
 6   descripcion_grupo_usuario       object                   
 7   grupo_usuario_especifico        int64                    
 8   descripcion_grupo_usuario_espe  object                   
 9   ordinal_de_tramo                int64                    
 10  cantidad_pasajeros              int64                    
 11  codigo_parada_origen            int64                    
 12

None

In [8]:
# Filtrar viajes de lunes a viernes
# 0=Lunes, 1=Martes, ..., 4=Viernes

df['dia_semana'] = df['fecha_evento'].dt.dayofweek
viajes_lun_vie = df[df['dia_semana'].isin([0,1,2,3,4])].copy()

print(f"Cantidad de viajes de lunes a viernes: {len(viajes_lun_vie)}")
display(viajes_lun_vie.head())

Cantidad de viajes de lunes a viernes: 10760006


,id_viaje,con_tarjeta,fecha_evento,tipo_viaje,descripcion_tipo_viaje,grupo_usuario,descripcion_grupo_usuario,grupo_usuario_especifico,descripcion_grupo_usuario_espe,ordinal_de_tramo,cantidad_pasajeros,codigo_parada_origen,cod_empresa,descrip_empresa,linea_codigo,dsc_linea,sevar_codigo,dia_semana
0,1,1,2022-03-01 00:00:02-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,1642,50,C.U.T.C.S.A.,446,151,3363,1
1,2,1,2022-03-01 00:00:06-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,1642,50,C.U.T.C.S.A.,446,151,3363,1
2,3,1,2022-03-01 00:02:01-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,3713,50,C.U.T.C.S.A.,73,158,8546,1
3,4,1,2022-03-01 00:09:40-03:00,5.0,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1,1,3217,50,C.U.T.C.S.A.,26,102,269,1
4,5,0,2022-03-01 00:07:12-03:00,3.0,CENTRICO,0,NO CORRESPONDE,0,NO CORRESPONDE,0,1,4014,50,C.U.T.C.S.A.,98,60,1483,1


In [9]:
# Calcular horario promedio por parada y línea
# Agrupamos por línea, parada y hora, y calculamos el promedio de horario por día

# Extraer hora de paso
viajes_lun_vie['hora'] = viajes_lun_vie['fecha_evento'].dt.hour
viajes_lun_vie['minuto'] = viajes_lun_vie['fecha_evento'].dt.minute

# Convertir hora:minuto a minutos desde medianoche para promediar
viajes_lun_vie['minutos_desde_medianoche'] = viajes_lun_vie['hora']*60 + viajes_lun_vie['minuto']

# Agrupar por línea, parada, día y hora, y calcular promedio de minutos
promedios = viajes_lun_vie.groupby(['dsc_linea','codigo_parada_origen','hora','dia_semana'])['minutos_desde_medianoche'].mean().reset_index()

# Convertir el promedio de minutos a horario (hh:mm)
promedios['horario'] = promedios['minutos_desde_medianoche'].apply(lambda x: f"{int(x//60):02d}:{int(x%60):02d}")

# Mostrar una muestra del resultado
display(promedios.head())

,dsc_linea,codigo_parada_origen,hora,dia_semana,minutos_desde_medianoche,horario
0,100,2008,0,2,14.000000,00:14
1,100,2008,5,0,349.333333,05:49
2,100,2008,5,1,352.500000,05:52
3,100,2008,5,2,349.500000,05:49
4,100,2008,6,0,412.400000,06:52


In [10]:
def filter_schedules_by_hour_difference(group):
    if group.empty:
        return group

    # Sort by time within each group
    group = group.sort_values(by='minutos_desde_medianoche').reset_index(drop=True)

    filtered_rows = []
    # Keep the first row by default
    if not group.empty:
        filtered_rows.append(group.iloc[0])
        last_kept_time = group.iloc[0]['minutos_desde_medianoche']

        for i in range(1, len(group)):
            current_row = group.iloc[i]
            current_time = current_row['minutos_desde_medianoche']
            # Check if current time is at least 60 minutes after the last kept time
            if current_time - last_kept_time >= 60:
                filtered_rows.append(current_row)
                last_kept_time = current_time

    return pd.DataFrame(filtered_rows)

# Apply the filtering function to each group
promedios_filtrados = promedios.groupby(['dsc_linea', 'codigo_parada_origen', 'dia_semana']).apply(filter_schedules_by_hour_difference, include_groups=False).reset_index()

# Display a sample of the result and comparison of counts
display(promedios_filtrados.head())
print(f"Cantidad de horarios antes de filtrar: {len(promedios)}")
print(f"Cantidad de horarios después de filtrar: {len(promedios_filtrados)}")

,dsc_linea,codigo_parada_origen,dia_semana,level_3,hora,minutos_desde_medianoche,horario
0,100,2008,0,0,5,349.333333,05:49
1,100,2008,0,1,6,412.400000,06:52
2,100,2008,0,3,8,518.000000,08:38
3,100,2008,0,5,10,637.000000,10:37
4,100,2008,0,7,12,759.222222,12:39


Cantidad de horarios antes de filtrar: 894108
Cantidad de horarios después de filtrar: 606795


In [11]:
promedios_filtrados.head(0)

,dsc_linea,codigo_parada_origen,dia_semana,level_3,hora,minutos_desde_medianoche,horario


In [12]:
# Construir dataframe final y exportar a CSV
# El dataframe tendrá: cod_Linea, horario, id_parada

df_horario_teorico = promedios_filtrados[['dsc_linea', 'horario', 'codigo_parada_origen']].copy()
df_horario_teorico = df_horario_teorico.rename(columns={
    'dsc_linea': 'dsc_linea',
    'codigo_parada_origen': 'id_parada'
})


# Exportar a CSV
output_path = '/content/drive/MyDrive/AAGrafos/Proyecto/horario_teorico/horario_teorico2.csv'
df_horario_teorico.to_csv(output_path, index=False)
print(f"Archivo guardado en {output_path}")
display(df_horario_teorico.head())

Archivo guardado en /content/drive/MyDrive/AAGrafos/Proyecto/horario_teorico/horario_teorico2.csv


,dsc_linea,horario,id_parada
0,100,05:49,2008
1,100,06:52,2008
2,100,08:38,2008
3,100,10:37,2008
4,100,12:39,2008


In [13]:
print(df_horario_teorico['dsc_linea'].unique())
print(len(df_horario_teorico['dsc_linea']))

['100' '102' '103' '104' '105' '106' '109' '110' '111' '112' '113' '115'
 '116' '117' '121' '124' '124 SD' '125' '127' '128' '130' '133' '135'
 '137' '140' '141' '142' '143' '144' '145' '147' '148' '149' '150' '151'
 '155' '156' '157' '158' '163' '169' '17' '174' '175' '180' '181' '182'
 '183' '185' '186' '187' '188' '191' '192' '195' '199' '2' '21' '300'
 '306' '316' '328' '329' '330' '370' '396' '402' '404' '405' '407' '409'
 '427' '456' '494' '495' '505' '522' '524' '526' '538' '546' '582' '60'
 '62' '64' '71' '76' '79' 'CE1' 'D10' 'D11' 'D5' 'D8' 'D9' 'DE1' 'E14' 'G'
 'G10' 'G11' 'G3' 'G6' 'G8' 'L1' 'L12' 'L13' 'L14' 'L15' 'L16' 'L17' 'L18'
 'L19' 'L2' 'L20' 'L22' 'L23' 'L24' 'L25' 'L26' 'L28' 'L29' 'L3' 'L30'
 'L31' 'L32' 'L33' 'L34' 'L35' 'L36' 'L38' 'L39' 'L4' 'L41' 'L46' 'L5'
 'L6' 'L7']
606795


In [14]:
# Calcular el tiempo mediano (en minutos desde medianoche) para cada parada en cada línea,
# esto ayudará a inferir el orden temporal de las paradas para una línea.
stop_order_info = promedios_filtrados.groupby(['dsc_linea', 'codigo_parada_origen'])['minutos_desde_medianoche'].median().reset_index()

# Ordenar las paradas dentro de cada línea basándose en su tiempo mediano de paso.
stop_order_info_sorted = stop_order_info.sort_values(by=['dsc_linea', 'minutos_desde_medianoche'])

# Crear el DataFrame 'df_recorridos' donde cada fila es una línea y su secuencia ordenada de paradas únicas.
df_recorridos = (
    stop_order_info_sorted.groupby('dsc_linea')['codigo_parada_origen']
    .apply(list) # Convertir la serie ordenada de paradas en una lista para cada línea.
    .reset_index(name='recorrido')
)

print("DataFrame de recorridos (secuencia de paradas por línea):")
display(df_recorridos.head())
print(f"Número total de líneas con secuencia de paradas: {len(df_recorridos)}")

DataFrame de recorridos (secuencia de paradas por línea):


,dsc_linea,recorrido
0,100,"[4758, 3914, 4718, 4593, 3196, 5280, 2018, 202..."
1,102,"[4763, 4764, 2754, 2716, 2759, 2751, 2753, 358..."
2,103,"[4007, 4041, 2109, 4764, 4771, 4763, 4772, 492..."
3,104,"[4769, 4770, 5352, 5354, 5363, 4598, 5305, 530..."
4,105,"[4041, 4765, 4763, 4769, 4771, 4929, 4770, 477..."


Número total de líneas con secuencia de paradas: 136


In [15]:
df_recorridos.to_csv('/content/drive/MyDrive/AAGrafos/Proyecto/horario_teorico/recorridos2.csv', index=False)